In [ ]:
import os 
import json 
from langchain_core.documents import Document
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_mistralai import ChatMistralAI
from langchain_community.vectorstores import FAISS
from langchain_core.prompts import ChatPromptTemplate
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.output_parsers import StrOutputParser
from dotenv import load_dotenv

# for developing purpose
# from langchain_ollama import ChatOllama

### laoding the configs

In [ ]:
load_dotenv()
MISTRAL_API_KEY= os.getenv("MISTRAL_API_KEY")
DATASET_PATH = "papers.json"
# there might be chances that the rate-limit exceeds while testing the notebook
llm = ChatMistralAI(model="mistral-small-latest", api_key=MISTRAL_API_KEY)
# for developing purpose
# llm = ChatOllama(model="llama3.1")

### Data Preparationg

In [36]:
def load_and_preprocess_papers(file_path: str) -> list[Document]:
    with open(file_path, "r", encoding="utf-8") as f:
        papers = json.load(f)
    raw_documents = []
    for paper in papers:
        title = paper.get('title', '')
        abstract = paper.get('abstract', '')
        keywords = paper.get('keywords', '')
        full_text = paper.get('full_text', '')
        if isinstance(keywords, list):
            keywords = ", ".join(keywords)
        page_content = f"Title: {title}\nKeywords: {keywords}\nAbstract: {abstract}\n\nFUll text: {full_text}"
        metadata = {
            "title" : title,
            "year" : paper.get('year', 0),
        }
        raw_documents.append(Document(page_content=page_content, metadata=metadata))
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=500,
        chunk_overlap=50,
        separators=['\n\n','\n', '.', ' ']
    )
    chunked_documents = text_splitter.split_documents(raw_documents)
    return chunked_documents

### building the vector store

In [37]:
def build_vector_store(documents: list[Document]) -> FAISS:
    embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
    vector_store = FAISS.from_documents(documents=documents, embedding=embeddings)
    return vector_store


### Retrieval and reranking the documents on the basis of latest years, then de-duplicating the results and returning the top-k (if left after deduplicating) documents.

In [38]:
def retrieve_and_rerank(query: str, vector_store: FAISS, k : int = 2, fetch_k:int =20) -> list[tuple[Document, float]]:
    results = vector_store.similarity_search_with_score(query=query, k=fetch_k)
    reranked_results = sorted(results, 
                              key=lambda x: (-x[0].metadata.get("year", 0), x[1])
                              )

    unique_papers = []
    seen_titles = set()

    for doc, score in reranked_results:
        title = doc.metadata.get('title', '')
        if title not in seen_titles:
            seen_titles.add(title)
            unique_papers.append((doc,score))
            if len(unique_papers) == k:
                break
    return unique_papers

### Main Pipeline

In [39]:
if os.path.exists(DATASET_PATH):
    documents = load_and_preprocess_papers(DATASET_PATH)
    vector_store=  build_vector_store(documents=documents)
    query = "Quantum computing in Cryptography"
    top_papers = retrieve_and_rerank(query=query, vector_store=vector_store, k=5)

    for rank, (doc, store) in enumerate(top_papers):
        print(f"{rank}. {[doc.metadata.get('year', 0)]} {doc.metadata.get('title' , '')} (Distance: {store:.4f})")
else:
    raise ValueError("Document not found.")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1686.98it/s]


0. [2024] Post-Quantum Cryptography and Quantum Key Distribution: A Survey (Distance: 0.5005)
1. [2023] Recent Advances in Generative AI for Text Synthesis (Distance: 1.9791)
2. [2022] Quantum Cryptography: A Comprehensive Review (Distance: 0.3480)
3. [2021] A Primer on Classical Cryptography Systems (Distance: 0.6565)
4. [2017] Attention Is All You Need (Distance: 1.8316)


### Altough the generation part is not mentioned in the Rubric, still for a safer side i have implemented a basic generator for the RAG Application that we are trying to build.

In [56]:
try:
    prompt = ChatPromptTemplate.from_template(
        """You are a Cryptography Researcher with 10 years of experience, you task is to provide the correct reponse for the user query and relevant context provided. If you analyze that the context is not relevant enough to answer the query, then simply return 'I am not capable to answer this query at the moment.'.env
        <CONTEXT>\n\n{context}\n\n</CONTEXT>\n\n
        User Query:\n{query}
        """
    )
    context = "\n\n".join([doc.page_content for doc, _ in top_papers])
    chain = prompt | llm | StrOutputParser()
    for chunk in chain.stream({"context" : context, "query" : query}):
        print(chunk, end="", flush=True)
    print()
except Exception as e:
    print(f"Error: {e}")

**Quantum Computing in Cryptography: A Threat and an Opportunity**

Quantum computing poses a significant threat to classical cryptographic systems like RSA and ECC, as mentioned in the provided context. However, it also presents opportunities for developing more secure cryptographic protocols.

The primary concern is that large-scale quantum computers could potentially break certain types of encryption, such as those relying on factorization (like RSA) or discrete logarithm problems (like ECC). This threat has led researchers to explore two main approaches: **Post-Quantum Cryptography (PQC)** and **Quantum Key Distribution (QKD)**.

**Key Points:**

1.  Quantum computers can potentially break certain types of encryption, making PQC and QKD essential for mitigating these risks.
2.  PQC involves mathematical algorithms designed to be secure against quantum computers, while QKD uses the principles of quantum mechanics to guarantee secure key exchange.
3.  Researchers are actively working